# Member 1: AI Skill Assessment & Skill Gap Engine
### Smart Academia–Industry Collaboration Portal
**Theme:** Smart Automation | **Technology:** Python + Google Colab

This notebook demonstrates:
1. Dynamic question assessment across Technical, Soft Skills, and Aptitude
2. Multi-tier evaluation (Technical, Soft Skills, Aptitude, Granular Skill Scores)
3. Automatic Skill Gap Identification (Strong >= 70%, Weak 40-69%, Missing < 40%)
4. Structured skill profile generation for database storage
5. Visual analytics for student skill readiness


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

print('Libraries loaded.')


In [ ]:
QUESTION_BANK = [
    {'id': 1, 'skill': 'Python', 'category': 'Technical', 'q': 'Which data structure is mutable in Python?', 'options': {'A': 'Tuple', 'B': 'List', 'C': 'String', 'D': 'Integer'}, 'correct': 'B'},
    {'id': 2, 'skill': 'Python', 'category': 'Technical', 'q': 'What does yield keyword do?', 'options': {'A': 'Terminates function', 'B': 'Generates values lazily', 'C': 'Pauses thread', 'D': 'Async call'}, 'correct': 'B'},
    {'id': 3, 'skill': 'SQL', 'category': 'Technical', 'q': 'Which clause filters aggregate results?', 'options': {'A': 'WHERE', 'B': 'GROUP BY', 'C': 'HAVING', 'D': 'ORDER BY'}, 'correct': 'C'},
    {'id': 4, 'skill': 'Machine Learning', 'category': 'Technical', 'q': 'Which is an unsupervised algorithm?', 'options': {'A': 'Linear Regression', 'B': 'K-Means', 'C': 'SVM', 'D': 'Random Forest'}, 'correct': 'B'},
    {'id': 5, 'skill': 'Cloud Computing', 'category': 'Technical', 'q': 'Which model provides runtime without managing servers?', 'options': {'A': 'IaaS', 'B': 'PaaS', 'C': 'SaaS', 'D': 'BaaS'}, 'correct': 'B'},
    {'id': 6, 'skill': 'Communication', 'category': 'Soft Skills', 'q': 'Best practice presenting architecture to business stakeholders?', 'options': {'A': 'Deep technical jargon', 'B': 'Clear impact and diagrams', 'C': 'No questions', 'D': 'No timeline'}, 'correct': 'B'},
    {'id': 7, 'skill': 'Quantitative Aptitude', 'category': 'Aptitude', 'q': 'Train at 54 km/h crosses 150m platform in 20s. Train length?', 'options': {'A': '100m', 'B': '150m', 'C': '200m', 'D': '250m'}, 'correct': 'B'}
]
print(f'Question bank ready with {len(QUESTION_BANK)} questions.')


In [ ]:
class ColabAssessmentEngine:
    def __init__(self, questions):
        self.questions = questions

    def evaluate(self, answers_dict):
        tech_corr, tech_tot = 0, 0
        soft_corr, soft_tot = 0, 0
        apt_corr, apt_tot = 0, 0
        skill_stats = {}

        for q in self.questions:
            qid = q['id']
            sel = answers_dict.get(qid, '').upper()
            is_corr = (sel == q['correct'])
            cat = q['category']
            sk = q['skill']

            if cat == 'Technical': tech_tot += 1; tech_corr += int(is_corr)
            elif cat == 'Soft Skills': soft_tot += 1; soft_corr += int(is_corr)
            elif cat == 'Aptitude': apt_tot += 1; apt_corr += int(is_corr)

            if sk not in skill_stats: skill_stats[sk] = {'c': 0, 't': 0}
            skill_stats[sk]['t'] += 1
            skill_stats[sk]['c'] += int(is_corr)

        pct = lambda c, t: round((c / t) * 100.0, 1) if t else 0.0
        tech_score = pct(tech_corr, tech_tot)
        soft_score = pct(soft_corr, soft_tot)
        apt_score = pct(apt_corr, apt_tot)
        overall = round((tech_score * 0.5) + (soft_score * 0.25) + (apt_score * 0.25), 1)

        strong, weak, missing = [], [], []
        skill_pcts = {}
        for sk, dat in skill_stats.items():
            score = pct(dat['c'], dat['t'])
            skill_pcts[sk] = score
            if score >= 70: strong.append(sk)
            elif score >= 40: weak.append(sk)
            else: missing.append(sk)

        return {
            'technical_score': tech_score,
            'soft_skill_score': soft_score,
            'aptitude_score': apt_score,
            'overall_readiness': overall,
            'skill_scores': skill_pcts,
            'strong_skills': strong,
            'weak_skills': weak,
            'missing_skills': missing
        }


In [ ]:
engine = ColabAssessmentEngine(QUESTION_BANK)
student_answers = {1: 'B', 2: 'B', 3: 'C', 4: 'A', 5: 'A', 6: 'B', 7: 'B'}
result = engine.evaluate(student_answers)
print('Evaluation Result:')
print(json.dumps(result, indent=2))


In [ ]:
skills = list(result['skill_scores'].keys())
scores = list(result['skill_scores'].values())
colors = ['#10b981' if s >= 70 else ('#f59e0b' if s >= 40 else '#ef4444') for s in scores]

plt.figure(figsize=(9, 4.5))
bars = plt.barh(skills, scores, color=colors)
plt.axvline(70, color='green', linestyle='--', label='Target Readiness (70%)')
plt.axvline(40, color='orange', linestyle='--', label='Minimum Baseline (40%)')
plt.title('Student AI Skill Evaluation & Gap Identification', fontsize=13)
plt.xlabel('Score (%)')
plt.xlim(0, 100)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()
